# Azure AI Search ជាមួយ NVIDIA NIM និងការរួមបញ្ចូល LlamaIndex

នៅក្នុងសៀវភៅកំណត់ត្រានេះ យើង​នឹងបង្ហាញពីរបៀបប្រើម៉ូឌែល AI របស់ NVIDIA និង LlamaIndex ដើម្បីបង្កើតបណ្តាញការបង្កើតដែលពង្រឹងដោយការទាញយក (Retrieval-Augmented Generation - RAG) យ៉ាងមានសមត្ថភាព។ យើង​នឹងប្រើ LLMs និង embeddings របស់ NVIDIA, បញ្ចូលពួកវាជាមួយ Azure AI Search ជាឃ្លាំងវ៉ិចទ័រ (vector store), ហើយអនុវត្ត RAG ដើម្បីពង្រឹងគុណភាពនិងប្រសិទ្ធភាពនៃការស្វែងរក។

## អត្ថប្រយោជន៍
- **ការអាចពង្រីកបាន**: ប្រើម៉ូឌែលភាសាធំរបស់ NVIDIA និង Azure AI Search សម្រាប់ការទាញយកដែលអាចពង្រីកបាន និងមានប្រសិទ្ធភាព។
- **ប្រសិទ្ធភាពថ្លៃដើម**: បង្កើតអុបទ៊ីម៉ាល់សម្រាប់ស្វែងរក និងទាញយកដោយបញ្ចូលឃ្លាំងវ៉ិចទ័រដែលមានប្រសិទ្ធភាព និងបច្ចេកវិទ្យាស្វែងរករួម។
- **ប្រសិទ្ធិភាពខ្ពស់**: បញ្ចូល LLMs ដែលមានអំណាចជាមួយការស្វែងរកតាមវ៉ិចទ័រ ដើម្បីឱ្យបានចម្លើយលឿន និងត្រឹមត្រូវជាងមុន។
- **គុណភាព**: រក្សាគុណភាពស្វែងរកខ្ពស់ដោយភ្ជាប់ចម្លើយរបស់ LLM ទៅនឹងឯកសារដែលបានទាញយកដែលពាក់ព័ន្ធ។

## លក្ខខណ្ឌមុន
- 🐍 Python 3.9 ឬខ្ពស់ជាងនេះ
- 🔗 [សេវាកម្ម Azure AI Search](https://learn.microsoft.com/azure/search/)
- 🔗 កូនសោ API របស់ NVIDIA សម្រាប់ការចូលប្រើ LLMs និង Embeddings របស់ NVIDIA តាមរយៈមីក្រូសេវាកម្ម NVIDIA NIM

## លក្ខណៈពិសេសដែលបានគ្របដណ្តប់
- ✅ ការរួមបញ្ចូល NVIDIA LLM (យើងនឹងប្រើ [Phi-3.5-MOE](https://build.nvidia.com/microsoft/phi-3_5-moe))
- ✅ Embeddings របស់ NVIDIA (យើងនឹងប្រើ [nv-embedqa-e5-v5](https://build.nvidia.com/nvidia/nv-embedqa-e5-v5))
- ✅ របៀបទាញយកកម្រិតខ្ពស់របស់ Azure AI Search
- ✅ ការធ្វើសន្ទស្សន៍ឯកសារជាមួយ LlamaIndex
- ✅ RAG ដោយប្រើ Azure AI Search និង LlamaIndex ជាមួយ LLMs របស់ NVIDIA

ចាប់ផ្តើម!


In [ ]:
!pip install azure-search-documents==11.5.1
!pip install --upgrade llama-index
!pip install --upgrade llama-index-core
!pip install --upgrade llama-index-readers-file
!pip install --upgrade llama-index-llms-nvidia
!pip install --upgrade llama-index-embeddings-nvidia
!pip install --upgrade llama-index-postprocessor-nvidia-rerank
!pip install --upgrade llama-index-vector-stores-azureaisearch
!pip install python-dotenv

## ការតំឡើង និងតម្រូវការ
បង្កើតបរិយាកាស Python ដោយប្រើកំណែ Python >3.10។

## ការចាប់ផ្ដើម!


ដើម្បីចាប់ផ្តើម អ្នកត្រូវការឲ្យមាន `NVIDIA_API_KEY` ដើម្បីប្រើម៉ូឌែល NVIDIA AI Foundation:
1) បង្កើតគណនីឥតគិតថ្លៃជាមួយ [NVIDIA](https://build.nvidia.com/explore/discover).
2) ចុចលើម៉ូឌែលដែលអ្នកជ្រើស។
3) ក្រោម Input, ជ្រើសផ្ទាំង Python, ហើយចុច **Get API Key** និងបន្ទាប់មកចុច **Generate Key**.
4) ចម្លង និងរក្សាទុកកូនសោដែលបានបង្កើតជា NVIDIA_API_KEY។ ពីទីនោះ អ្នកគួរតែមានការចូលដំណើរការទៅកាន់ endpoints។


In [3]:
import getpass
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

if not os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
    nvidia_api_key = getpass.getpass("Enter your NVIDIA API key: ")
    assert nvidia_api_key.startswith("nvapi-"), f"{nvidia_api_key[:5]}... is not a valid key"
    os.environ["NVIDIA_API_KEY"] = nvidia_api_key


## ឧទាហរណ៍ RAG ដែលប្រើ LLM និង Embedding
### 1) ចាប់ផ្តើម LLM
`llama-index-llms-nvidia`, ដែលគេស្គាល់ថា ជា ឧបករណ៍ភ្ជាប់ LLM របស់ NVIDIA, អនុញ្ញាតឱ្យអ្នកភ្ជាប់ និងបង្កើតពីម៉ូឌែលដែលត្រូវគ្នា ដែលមាននៅក្នុងកាតាឡុក API របស់ NVIDIA។ សូមមើលទីនេះសម្រាប់បញ្ជីម៉ូឌែល chat completion: https://build.nvidia.com/search?term=Text-to-Text

នៅទីនេះយើងនឹងប្រើ **mixtral-8x7b-instruct-v0.1**


In [75]:
from llama_index.core import Settings
from llama_index.llms.nvidia import NVIDIA

# Here we are using mixtral-8x7b-instruct-v0.1 model from API Catalog
Settings.llm = NVIDIA(model="microsoft/phi-3.5-moe-instruct", api_key=os.getenv("NVIDIA_API_KEY"))

### 2) ចាប់ផ្តើមការបង្កើត Embedding
`llama-index-embeddings-nvidia`, ដែលក៏ត្រូវបានហៅថា ជា​កម្មវិធីភ្ជាប់ Embeddings របស់ NVIDIA, អនុញ្ញាតឲ្យអ្នកភ្ជាប់ទៅនឹង និងបង្កើតពីម៉ូឌែល​ដែលសមស្រប ដែលមាននៅលើបញ្ជី API របស់ NVIDIA។ យើងបានជ្រើស `nvidia/nv-embedqa-e5-v5` ជា​ម៉ូឌែល Embedding។ សូមមើលនៅទីនេះសម្រាប់បញ្ជីម៉ូឌែល embedding សម្រាប់អត្ថបទ: https://build.nvidia.com/nim?filters=usecase%3Ausecase_text_to_embedding%2Cusecase%3Ausecase_image_to_embedding


In [6]:
from llama_index.embeddings.nvidia import NVIDIAEmbedding

Settings.embed_model = NVIDIAEmbedding(model="nvidia/nv-embedqa-e5-v5", api_key=os.getenv("NVIDIA_API_KEY"))

### 3) បង្កើតឃ្លាំងវ៉ិចទ័រ Azure AI Search


In [76]:
import logging
import sys
import os
import getpass
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from IPython.display import Markdown, display
from llama_index.vector_stores.azureaisearch import AzureAISearchVectorStore, IndexManagement


search_service_api_key = os.getenv('AZURE_SEARCH_ADMIN_KEY') or getpass.getpass('Enter your Azure Search API key: ')
search_service_endpoint = os.getenv('AZURE_SEARCH_SERVICE_ENDPOINT') or getpass.getpass('Enter your Azure Search service endpoint: ')
search_service_api_version = "2024-07-01"
credential = AzureKeyCredential(search_service_api_key)

# Index name to use
index_name = "llamaindex-nvidia-azureaisearch-demo"

# Use index client to demonstrate creating an index
index_client = SearchIndexClient(
    endpoint=search_service_endpoint,
    credential=credential,
)

# Use search client to demonstrate using existing index
search_client = SearchClient(
    endpoint=search_service_endpoint,
    index_name=index_name,
    credential=credential,
)

In [ ]:
vector_store = AzureAISearchVectorStore(
    search_or_index_client=index_client,
    index_name=index_name,
    index_management=IndexManagement.CREATE_IF_NOT_EXISTS,
    id_field_key="id",
    chunk_field_key="chunk",
    embedding_field_key="embedding",
    embedding_dimensionality=1024, # dimensionality for nv-embedqa-e5-v5 model
    metadata_string_field_key="metadata",
    doc_id_field_key="doc_id",
    language_analyzer="en.lucene",
    vector_algorithm_type="exhaustiveKnn",
    # compression_type="binary" # Option to use "scalar" or "binary". NOTE: compression is only supported for HNSW
)

### 4) ផ្ទុកឯកសារ, បំបែកជាចំណែក និងផ្ទុកឡើង


In [20]:
from llama_index.core import SimpleDirectoryReader, StorageContext, VectorStoreIndex
from llama_index.core.text_splitter import TokenTextSplitter

# Configure text splitter (nv-embedqa-e5-v5 model has a limit of 512 tokens per input size)
text_splitter = TokenTextSplitter(separator=" ", chunk_size=500, chunk_overlap=10)

# Load documents
documents = SimpleDirectoryReader(
    input_files=["data/txt/state_of_the_union.txt"]
).load_data()
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Create index with text splitter
index = VectorStoreIndex.from_documents(
    documents,
    transformations=[text_splitter],
    storage_context=storage_context,
)

### 5) បង្កើតម៉ាស៊ីន​ស្វែងរក​សំណួរ ដើម្បី​សួរព័ត៌មាន​ពីទិន្នន័យ​របស់អ្នក

នេះ​ជា​សំណួរ​មួយ​ដែល​ប្រើ​ការ​ស្វែងរក​វិចទ័រ​បរិសុទ្ធ​នៅ​ក្នុង Azure AI Search និង​ធ្វើឲ្យ​ចម្លើយ​ផ្អែក​លើ LLM របស់​យើង (Phi-3.5-MOE)


In [69]:
query_engine = index.as_query_engine()
response = query_engine.query("Who did the speaker mention as being present in the chamber?")
display(Markdown(f"{response}"))

 The speaker mentioned the Ukrainian Ambassador to the United States, along with other members of Congress, the Cabinet, and various officials such as the Vice President, the First Lady, and the Second Gentleman, as being present in the chamber.

នេះជាសំណួរស្វែងរកមួយដែលប្រើការស្វែងរកផ្សំក្នុង Azure AI Search.


In [70]:
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.vector_stores.types import VectorStoreQueryMode
from IPython.display import Markdown, display
from llama_index.core.schema import MetadataMode

# Initialize hybrid retriever and query engine
hybrid_retriever = index.as_retriever(vector_store_query_mode=VectorStoreQueryMode.HYBRID)
hybrid_query_engine = RetrieverQueryEngine(retriever=hybrid_retriever)

# Query execution
query = "What were the exact economic consequences mentioned in relation to Russia's stock market?"
response = hybrid_query_engine.query(query)

# Display the response
display(Markdown(f"{response}"))
print("\n")

# Print the source nodes
print("Source Nodes:")
for node in response.source_nodes:
    print(node.get_content(metadata_mode=MetadataMode.LLM))

 The Russian stock market experienced a significant drop, losing 40% of its value. Additionally, trading had to be suspended due to the ongoing situation.



Source Nodes:
file_path: data\txt\state_of_the_union.txt

building a coalition of other freedom-loving nations from Europe and the Americas to Asia and Africa to confront Putin. 

I spent countless hours unifying our European allies. We shared with the world in advance what we knew Putin was planning and precisely how he would try to falsely justify his aggression.  

We countered Russia’s lies with truth.   

And now that he has acted the free world is holding him accountable. 

Along with twenty-seven members of the European Union including France, Germany, Italy, as well as countries like the United Kingdom, Canada, Japan, Korea, Australia, New Zealand, and many others, even Switzerland. 

We are inflicting pain on Russia and supporting the people of Ukraine. Putin is now isolated from the world more than ever. 

Together with our allies –we are right now enforcing powerful economic sanctions. 

We are cutting off Russia’s largest banks from the international financial system.  



#### Vector Search Analysis
ការឆ្លើយតបរបស់ LLM បានចាប់យកយ៉ាងត្រឹមត្រូវលើផលវិបាកសេដ្ឋកិច្ចសំខាន់ៗដែលបានរៀបរាប់នៅក្នុងអត្ថបទប្រភពទាក់ទងនឹងទីផ្សារហ៊ុនរុស្ស៊ី។ ជាក់លាក់ វាបាននិយាយថាទីផ្សារហ៊ុនរុស្ស៊ីបានធ្លាក់ចុះយ៉ាងខ្លាំង បាត់បង់តម្លៃ 40% ហើយការជួញដូរត្រូវបានផ្អាកដោយសារស្ថានភាពដែលកំពុងបន្ត។ ការឆ្លើយនេះស្របទៅនឹងព័ត៌មានដែលបានផ្តល់ក្នុងប្រភព បង្ហាញថា LLM បានស្គាល់ និងសង្ខេបព័ត៌មានពាក់ព័ន្ធអំពីផលប៉ះពាល់លើទីផ្សារហ៊ុនដោយសារ សកម្មភាពរបស់រុស្ស៊ី និងការដាក់ទណ្ឌកម្មដែលបានអនុវត្ត បានយ៉ាងត្រឹមត្រូវ។

#### Source Nodes Commentary
ចំណុចប្រភពបានផ្តល់ការរៀបរាប់លម្អិតអំពីផលវិបាកសេដ្ឋកិច្ចដែលរុស្ស៊ីបានប្រឈមមុខដោយសារការដាក់ទណ្ឌកម្មអន្តរជាតិ។ អត្ថបទបានលើកឡើងថាទីផ្សារហ៊ុនរុស្ស៊ីបាត់បង់ 40% នៃតម្លៃ ហើយការជួញដូរត្រូវបានផ្អាក។ បន្ថែមពីនេះ វាក៏បានរៀបរាប់ពីផលវិបាកសេដ្ឋកិច្ចផ្សេងទៀត ដូចជា ការធ្លាក់តម្លៃរបស់រ៉ូបល និងការបែកចេញយ៉ាងទូលំទូលាយនៃសេដ្ឋកិច្ចរុស្ស៊ី។ ការឆ្លើយរបស់ LLM បានរើសចំណុចសំខាន់ៗពីចំណុចទាំងនេះបានយ៉ាងមានប្រសិទ្ធភាព ដោយផ្តោតលើផលប៉ះពាល់លើទីផ្សារហ៊ុនដូចដែលបានស្នើដោយសំណើ។


ឥឡូវនេះ យើងមកមើលសំណួរមួយដែល Hybrid Search មិនបានផ្តល់ចម្លើយដែលមានមូលដ្ឋានល្អ:


In [71]:
# Query execution
query = "What was the precise date when Russia invaded Ukraine?"
response = hybrid_query_engine.query(query)

# Display the response
display(Markdown(f"{response}"))
print("\n")

# Print the source nodes
print("Source Nodes:")
for node in response.source_nodes:
    print(node.get_content(metadata_mode=MetadataMode.LLM))


 The provided context does not specify the exact date of Russia's invasion of Ukraine. However, it does mention that the events discussed are happening in the current era and that the actions taken are in response to Putin's aggression. For the precise date, one would need to refer to external sources or historical records.



Source Nodes:
file_path: data\txt\state_of_the_union.txt

our forces are not engaged and will not engage in conflict with Russian forces in Ukraine.  

Our forces are not going to Europe to fight in Ukraine, but to defend our NATO Allies – in the event that Putin decides to keep moving west.  

For that purpose we’ve mobilized American ground forces, air squadrons, and ship deployments to protect NATO countries including Poland, Romania, Latvia, Lithuania, and Estonia. 

As I have made crystal clear the United States and our Allies will defend every inch of territory of NATO countries with the full force of our collective power.  

And we remain clear-eyed. The Ukrainians are fighting back with pure courage. But the next few days weeks, months, will be hard on them.  

Putin has unleashed violence and chaos.  But while he may make gains on the battlefield – he will pay a continuing high price over the long run. 

And a proud Ukrainian people, who have known 30 years  of independence,

### Hybrid Search: វិភាគចម្លើយរបស់ LLM
ចម្លើយរបស់ LLM ក្នុងឧទាហរណ៍ Hybrid Search បានបញ្ជាក់ថា បរិបទដែលបានផ្ដល់មិនបានកំណត់ថ្ងៃខែឆ្នាំជាក់លាក់នៃការវាយប្រហាររបស់រុស្ស៊ីលើអ៊ុយក្រែន។ ចម្លើយនេះបង្ហាញថា LLM កំពុងប្រើព័ត៌មានដែលមាននៅក្នុងឯកសារមូលដ្ឋាន ប៉ុន្តែទទួលស្គាល់ពីការខ្វះខាតនៃព័ត៌មានលម្អិតដែលច្បាស់លាស់ក្នុងអត្ថបទ។

ចម្លើយនេះត្រឹមត្រូវក្នុងការបញ្ជាក់ថា បរិបទបានចង្អុលចង្អៀតពីព្រឹត្តិការណ៍ដែលទាក់ទងនឹងការវាយប្រហាររបស់រុស្ស៊ី ប៉ុន្តែមិនអាចកំណត់ថ្ងៃខែឆ្នាំនៃការវាយប្រហារ secara ជាក់លាក់បាន។ នេះបង្ហាញពីសមត្ថភាពរបស់ LLM ក្នុងការយល់អត្ថន័យពីព័ត៌មានដែលបានផ្តល់ ខណៈពេលដែលវាដឹងថាមានចន្លោះខ្វះខាតក្នុងមាតិកា។ LLM បានជំរុញអ្នកប្រើឲ្យស្វែងរកប្រភពខាងក្រៅ ឬកំណត់ត្រាប្រវត្តិសាស្ត្រសម្រាប់ថ្ងៃខែឆ្នាំជាក់លាក់ បង្ហាញពីភាពប្រុងប្រយ័ត្ននៅពេលព័ត៌មានមិនពេញលេញ។

### វិភាគចំណុចប្រភព
ចំណុចប្រភពក្នុងឧទាហរណ៍ Hybrid Search មានចំណែកពីសុន្ទរកថាមួយ ដែលពិភាក្សាពាក់ព័ន្ធនឹងការឆ្លើយតបរបស់សហរដ្ឋអាមេរិកចំពោះសកម្មភាពរបស់រុស្ស៊ីនៅអ៊ុយក្រែន។ ចំណុចទាំងនេះលើកឡើងពីផលប៉ះពាល់នយោបាយអន្តរជាតិក្នុងវិសាលទូលំទូលាយ និងជំហានដែលសហរដ្ឋអាមេរិក និងមិត្តរួមបានអនុវត្តក្នុងការឆ្លើយតបចំពោះការវាយប្រហារ ប៉ុន្តែពួកវាមិនបានذكرថ្ងៃខែឆ្នាំជាក់លាក់នៃការវាយប្រហារ។ វានេះសម្របសម្រួលជាមួយចម្លើយរបស់ LLM ដែលបានកំណត់យ៉ាងត្រឹមត្រូវថាបរិបទខ្វះព័ត៌មានថ្ងៃខែឆ្នាំដែលច្បាស់។


In [72]:
# Initialize hybrid retriever and query engine
semantic_reranker_retriever = index.as_retriever(vector_store_query_mode=VectorStoreQueryMode.SEMANTIC_HYBRID)
semantic_reranker_query_engine = RetrieverQueryEngine(retriever=semantic_reranker_retriever)

# Query execution
query = "What was the precise date when Russia invaded Ukraine?"
response = semantic_reranker_query_engine.query(query)

# Display the response
display(Markdown(f"{response}"))
print("\n")

# Print the source nodes
print("Source Nodes:")
for node in response.source_nodes:
    print(node.get_content(metadata_mode=MetadataMode.LLM))


 The provided context does not specify the exact date of Russia's invasion of Ukraine. However, it mentions that the event occurred six days before the speech was given. To determine the precise date, one would need to know the date of the speech.



Source Nodes:
file_path: data\txt\state_of_the_union.txt

Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. 

He thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. 

He met the Ukrainian people. 

From President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world. 

### Hybrid w/Reranking: LLM Response Analysis
នៅក្នុងឧទាហរណ៍ Hybrid w/Reranking នេះ ចម្លើយពី LLM បានផ្តល់บริบทបន្ថែមដោយចំណាំថាព្រឹត្តិការណ៍បានកើតឡើងមុនសុន្ទរកថា ដែលបានផ្តល់ជូន ៦ ថ្ងៃ។ នេះបង្ហាញថា LLM អាចសន្មត់កាលបរិច្ឆេទនៃការវាយប្រហារបានដោយយោងទៅលើពេលវេលានៃសុន្ទរកថា ទោះបីវានៅតែត្រូវការកាលបរិច្ឆេទត្រឹមត្រូវនៃសុន្ទរកថាដើម្បីទទួលបានភាពទន់ច្បាស់កាន់តែច្បាស់។

ចម្លើយនេះបង្ហាញពីសមត្ថភាពដែលបានប្រសើរឡើងក្នុងការប្រើសញ្ញាបរិបទដើម្បីផ្តល់ចម្លើយដែលមានព័ត៌មានច្រើនជាងមុន។ វានៅឡើយក៏លើកសម្គាល់អត្ថប្រយោជន៍នៃ reranking ដែលអនុញ្ញាតឱ្យ LLM ចូលដល់ និងផ្តល់អាទិភាពដល់ព័ត៌មានពាក់ព័ន្ធជាង ដើម្បីផ្តល់ការប៉ាន់ស្មានដែលជិតនឹងព័ត៌មានដែលចង់បាន (ឧ. កាលបរិច្ឆេទនៃការវាយប្រហារ)។

### Source Nodes Analysis
ចំណុចប្រភពក្នុងឧទាហរណ៍នេះរួមបញ្ចូលការដកស្រង់ដែលយោងទៅលើពេលវេលានៃការវាយប្រហាររបស់រុស្ស៊ី ដែលបានចាត់ចែងពិសេសថាវាបានកើតឡើងមុនសុន្ទរកថា ៦ ថ្ងៃ។ ទោះបីជា​កាលបរិច្ឆេទដែលត្រឹមត្រូវនៅតែមិនបានបញ្ជាក់យ៉ាងច្បាស់ ក៏ដោយ ចំណុចទាំងនេះបានផ្តល់បរិបទពេលវេលាដែលអនុញ្ញាតឱ្យ LLM ផ្តល់ចម្លើយដែលមានលក្ខណៈលម្អិតជាងមុន។ ការរួមបញ្ចូលព័ត៌មានលម្អិតនេះបង្ហាញថា reranking អាចធ្វើឱ្យសមត្ថភាពរបស់ LLM ក្នុងការដកយក និងសន្មត់ព័ត៌មានពីបរិបទដែលបានផ្តល់កាន់តែប្រសើរឡើង ដោយនាំឱ្យមានចម្លើយដែលត្រឹមត្រូវ និងមានព័ត៌មានច្រើនជាងមុន។


**ចំណាំ:**
នៅក្នុងសៀវភៅកំណត់ត្រានេះ យើងបានប្រើសេវាមីក្រូ NVIDIA NIM ពីកាតាឡុក API របស់ NVIDIA.
API ខាងលើ `NVIDIA (llms)`, `NVIDIAEmbedding`, និង [Azure AI Search ស្វែងរកហ៊ីប្រ៊ីដសេម៉ាន្ទិក (ការរៀបលំដាប់ឡើងវិញ​ដែលមាននៅក្នុង)](https://learn.microsoft.com/azure/search/semantic-search-overview). ចំណាំថា API ខាងលើទាំងនេះអាចគាំទ្រសេវាមីក្រូដែលផ្ទុកដោយខ្លួនឯងផងដែរ. 

**ឧទាហរណ៍:**
```python
NVIDIA(model="meta/llama3-8b-instruct", base_url="http://your-nim-host-address:8000/v1")


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាកម្មបកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ខណៈយើងខិតខំរកភាពត្រឹមត្រូវ សូមយកចិត្តទុកដាក់ថាការបកប្រែដោយស្វ័យប្រវត្តិនេះអាចមានកំហុស ឬមិនម៉ត់ចត់។ ឯកសារដើមនៅក្នុងភាសាមាតុភូមិគួរត្រូវបានចាត់ទុកថាជាប្រភពយោងផ្លូវការ។ សម្រាប់ព័ត៌មានដែលមានសារៈសំខាន់ សូមណែនាំឱ្យបកប្រែដោយអ្នកបកប្រែវិជ្ជាជីវៈមនុស្ស។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសណាមួយដែលកើតឡើងពីការប្រើប្រាស់ការបកប្រែនេះ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
